In [1]:
# !pip install tabulate

In [2]:
import json
import textwrap
import pandas as pd
from ipywidgets import widgets
from IPython.display import display
from ipywidgets import HBox

In [3]:
# Load the wandb table data
def load_wandb_data(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
    return pd.DataFrame(data['data'], columns=data['columns'])

In [4]:
def fixedwidth(text):
    """Wrap text to fixed width"""
    return "\n".join(textwrap.wrap(str(text), width=120, replace_whitespace=False))

def format_row(row):
    """Format a single row for display as a table"""
    from tabulate import tabulate
    
    data = [
        ["Step", row['Step']],
        ["Reward", row['Reward']],
        ["Input", fixedwidth(row['Input'])],
        ["Prompt", fixedwidth(row['Prompt'])], 
        ["Completion", fixedwidth(row['Completion'])],
    ]
    
    return tabulate(data, tablefmt='grid')

def present_row(row):
    """Display a formatted row as table"""
    print(format_row(row))

In [5]:
def create_browse_app(df):
    """Create an interactive widget to browse through the data"""
    def browse_data(i=0):
        row = df.iloc[i]
        present_row(row)

    index = widgets.IntText(value=0, description='Index:')
    left_button = widgets.Button(description='Previous')
    right_button = widgets.Button(description='Next')

    def on_left_button_clicked(b):
        if index.value > 0:
            index.value -= 1

    def on_right_button_clicked(b):
        if index.value < len(df) - 1:
            index.value += 1

    left_button.on_click(on_left_button_clicked)
    right_button.on_click(on_right_button_clicked)

    ui = HBox([left_button, index, right_button])
    out = widgets.interactive_output(browse_data, {'i': index})

    display(ui, out)

In [6]:
# Load and display your data
from pathlib import Path


# read all json files in the directory using pathlib
directory = "../wandb/run-20250309_135137-bfc398za/files/media/table"
files = list(Path(directory).glob("interactions_*.json"))

# load all json files
df = pd.concat([load_wandb_data(f) for f in files])
df.sort_values(by="Reward", ascending=False, inplace=True)
print(len(df))

# Create the interactive browser
create_browse_app(df)

7232


Output()